# 09b — Data prep: 5,000-user cohort with temporal train/test split

Scaled replacement for notebook 09 cohort / data prep:

- **No** median-distance cohort pick
- **No** Encoding A complement mirror (`5.5 − r`)
- Iterative core filtering (≥20 ratings per user & movie)
- Top **5,000** users by rating count
- Movie vocab = union of movies rated by the cohort (within filtered pool)
- Temporal **80/20** train/test split per user

| Part | Section |
|------|--------|
| Part 0 | Setup (shared constants with nb 09) |
| Part 1 | Iterative core filtering |
| Part 2 | Cohort selection (top 5k) + movie vocab |
| Part 3 | Temporal 80/20 split → `channel1`, `mask`, `test_labels` |
| Part 4 | Training-set personality stats (`mu_user`, `mu_global`) |
| Part 5 | Verification |

Outputs → `data/processed/` (overwrite same names).

## Part 0 — Setup

In [1]:
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent

rating_path = root / "data" / "rating.csv"
out_dir = root / "data" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)

assert rating_path.exists(), f"Missing {rating_path}"

RATING_LEVELS = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0], dtype=np.float64)
K = len(RATING_LEVELS)
rating_to_idx = {float(r): i for i, r in enumerate(RATING_LEVELS)}

CHUNK_SIZE = 1_000_000
N_USERS_COHORT = 5_000
MIN_RATINGS = 20
TRAIN_FRAC = 0.8
CSV_DTYPES = {"userId": "int32", "movieId": "int32", "rating": "float32"}
CSV_DTYPES_TS = {"userId": "int32", "movieId": "int32", "rating": "float32", "timestamp": "str"}

print(f"Project root: {root}")
print(f"Rating file:  {rating_path}")
print(f"Output dir:   {out_dir}")
print(f"K={K}, CHUNK_SIZE={CHUNK_SIZE:,}, N_USERS_COHORT={N_USERS_COHORT:,}, MIN_RATINGS={MIN_RATINGS}")

Project root: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys
Rating file:  /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/rating.csv
Output dir:   /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed
K=10, CHUNK_SIZE=1,000,000, N_USERS_COHORT=5,000, MIN_RATINGS=20


## Part 1 — Iterative core filtering

Stream once for raw counts, then iteratively drop users/movies with &lt; 20 ratings until stable.

In [2]:
print(f"Streaming {rating_path.name} for edges + raw counts …")
edges_u_chunks = []
edges_m_chunks = []
user_counts_acc = defaultdict(int)
movie_counts_acc = defaultdict(int)

for chunk in pd.read_csv(rating_path, usecols=["userId", "movieId"], dtype=CSV_DTYPES, chunksize=CHUNK_SIZE):
    edges_u_chunks.append(chunk["userId"].to_numpy(dtype=np.int32, copy=True))
    edges_m_chunks.append(chunk["movieId"].to_numpy(dtype=np.int32, copy=True))
    uc = chunk.groupby("userId")["movieId"].count()
    for uid, c in uc.items():
        user_counts_acc[int(uid)] += int(c)
    mc = chunk.groupby("movieId")["userId"].count()
    for mid, c in mc.items():
        movie_counts_acc[int(mid)] += int(c)

edges_u = np.concatenate(edges_u_chunks)
edges_m = np.concatenate(edges_m_chunks)
del edges_u_chunks, edges_m_chunks

user_counts = pd.Series(user_counts_acc, dtype=np.int64)
movie_counts = pd.Series(movie_counts_acc, dtype=np.int64)
print(f"Raw pool: {len(user_counts):,} users, {len(movie_counts):,} movies, {len(edges_u):,} ratings")

u_keep = user_counts.index.to_numpy(dtype=np.int32)
m_keep = movie_counts.index.to_numpy(dtype=np.int32)

iteration = 0
while True:
    iteration += 1
    mask = np.isin(edges_u, u_keep) & np.isin(edges_m, m_keep)
    u_sub = edges_u[mask]
    m_sub = edges_m[mask]

    uc = pd.Series(u_sub).value_counts()
    mc = pd.Series(m_sub).value_counts()

    new_u = uc[uc >= MIN_RATINGS].index.to_numpy(dtype=np.int32)
    new_m = mc[mc >= MIN_RATINGS].index.to_numpy(dtype=np.int32)

    n_drop_u = len(u_keep) - len(new_u)
    n_drop_m = len(m_keep) - len(new_m)
    u_keep, m_keep = new_u, new_m

    print(
        f"  iter {iteration}: users={len(u_keep):,}  movies={len(m_keep):,}  "
        f"(dropped {n_drop_u:,} users, {n_drop_m:,} movies)  edges_kept={int(mask.sum()):,}"
    )
    if n_drop_u == 0 and n_drop_m == 0:
        break

keep_users = set(int(x) for x in u_keep)
keep_movies = set(int(x) for x in m_keep)

mask = np.isin(edges_u, u_keep) & np.isin(edges_m, m_keep)
u_sub = edges_u[mask]
m_sub = edges_m[mask]
user_counts_core = pd.Series(u_sub).value_counts().sort_index()
movie_counts_core = pd.Series(m_sub).value_counts().sort_index()

print(f"\nStable core: {len(keep_users):,} users, {len(keep_movies):,} movies, {len(u_sub):,} ratings")
print(
    f"  user rating-count: min={int(user_counts_core.min())}, "
    f"median={float(user_counts_core.median()):.0f}, max={int(user_counts_core.max())}"
)
print(
    f"  movie rating-count: min={int(movie_counts_core.min())}, "
    f"median={float(movie_counts_core.median()):.0f}, max={int(movie_counts_core.max())}"
)

del edges_u, edges_m, u_sub, m_sub, mask, u_keep, m_keep


Streaming rating.csv for edges + raw counts …
Raw pool: 138,493 users, 26,744 movies, 20,000,263 ratings
  iter 1: users=138,493  movies=13,132  (dropped 0 users, 13,612 movies)  edges_kept=20,000,263
  iter 2: users=138,409  movies=13,132  (dropped 84 users, 0 movies)  edges_kept=19,933,089
  iter 3: users=138,409  movies=13,130  (dropped 0 users, 2 movies)  edges_kept=19,931,545
  iter 4: users=138,408  movies=13,130  (dropped 1 users, 0 movies)  edges_kept=19,931,507
  iter 5: users=138,408  movies=13,130  (dropped 0 users, 0 movies)  edges_kept=19,931,488

Stable core: 138,408 users, 13,130 movies, 19,931,488 ratings
  user rating-count: min=20, median=68, max=7540
  movie rating-count: min=20, median=215, max=67303


## Part 2 — Cohort selection (top 5,000) + movie vocabulary

In [3]:
cohort_series = user_counts_core.sort_values(ascending=False).head(N_USERS_COHORT)
cohort_user_ids = cohort_series.index.astype(int).to_numpy()
cohort_set = set(cohort_user_ids.tolist())

assert len(cohort_user_ids) == N_USERS_COHORT, (
    f"Expected {N_USERS_COHORT} users, got {len(cohort_user_ids)} — core pool too small?"
)

# Movie vocab = union of movies rated by cohort users, within filtered movie pool
print(f"Streaming {rating_path.name} for cohort × core-movie edges …")
vocab_set = set()
cohort_rating_counts = defaultdict(int)
for chunk in pd.read_csv(rating_path, usecols=["userId", "movieId"], dtype=CSV_DTYPES, chunksize=CHUNK_SIZE):
    sub = chunk[chunk["userId"].isin(cohort_set) & chunk["movieId"].isin(keep_movies)]
    if sub.empty:
        continue
    vocab_set.update(sub["movieId"].astype(int).tolist())
    for uid, c in sub.groupby("userId")["movieId"].count().items():
        cohort_rating_counts[int(uid)] += int(c)

movie_vocab = np.array(sorted(vocab_set), dtype=np.int64)
n_users = len(cohort_user_ids)
n_movies = len(movie_vocab)
movie_to_col = {int(mid): j for j, mid in enumerate(movie_vocab)}
user_to_row = {int(uid): i for i, uid in enumerate(cohort_user_ids)}

cohort_counts_arr = np.array([cohort_rating_counts[int(uid)] for uid in cohort_user_ids], dtype=np.int64)

np.save(out_dir / "cohort_user_ids.npy", cohort_user_ids)
np.save(out_dir / "movie_vocab.npy", movie_vocab)

print(f"Cohort size: {n_users:,}")
print(f"Movie vocab size: {n_movies:,}")
print(
    f"Cohort rating counts (in vocab): "
    f"min={int(cohort_counts_arr.min())}, "
    f"median={float(np.median(cohort_counts_arr)):.0f}, "
    f"max={int(cohort_counts_arr.max())}"
)
print(f"User 666 in cohort: {666 in cohort_set}")
print(f"Saved: {out_dir / 'cohort_user_ids.npy'}")
print(f"Saved: {out_dir / 'movie_vocab.npy'}")

Streaming rating.csv for cohort × core-movie edges …
Cohort size: 5,000
Movie vocab size: 13,129
Cohort rating counts (in vocab): min=619, median=871, max=7540
User 666 in cohort: False
Saved: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/cohort_user_ids.npy
Saved: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/movie_vocab.npy


## Part 3 — Temporal 80/20 split

Per user: sort by timestamp ascending; first `int(0.8 * n)` → train, rest → test.  
This is **not** a single global calendar cutoff — each user has their own train window, so train and test date ranges overlap globally.

**Observed ranges on the 5,000-user cohort × vocab (after running this cell):**

| | Earliest | Latest | n pairs |
|---|----------|--------|---------|
| Train (all train pairs) | 1996-05-17 | 2015-03-30 | ~4.12M |
| Test (all test pairs) | 1996-06-12 | 2015-03-31 | ~1.03M |

Per-user train window: start median ≈ 2004-08; end median ≈ 2006-08.  
Per-user test window: start median ≈ 2006-08; end median ≈ 2009-06.

- `channel1_softmax.npy` — train one-hots only `(n_users, n_movies, K)` float32
- `mask.npy` — 1 = train rating, 0 = test or unrated `(n_users, n_movies)` int8
- `test_labels.csv` — one row per test rating

In [4]:
print(f"Streaming {rating_path.name} for cohort × vocab ratings (+ timestamp) …")
user_rows: dict[int, list] = {int(uid): [] for uid in cohort_user_ids}

for chunk in pd.read_csv(
    rating_path,
    usecols=["userId", "movieId", "rating", "timestamp"],
    dtype=CSV_DTYPES_TS,
    chunksize=CHUNK_SIZE,
):
    sub = chunk[chunk["userId"].isin(cohort_set) & chunk["movieId"].isin(vocab_set)]
    if sub.empty:
        continue
    for row in sub.itertuples(index=False):
        uid = int(row.userId)
        user_rows[uid].append((str(row.timestamp), int(row.movieId), float(row.rating)))

channel1 = np.zeros((n_users, n_movies, K), dtype=np.float32)
mask = np.zeros((n_users, n_movies), dtype=np.int8)
test_records = []

n_train_pairs = 0
n_test_pairs = 0

for uid in cohort_user_ids:
    uid = int(uid)
    rows = user_rows[uid]
    # Sort by timestamp ascending (string ISO timestamps sort lexicographically)
    rows.sort(key=lambda x: x[0])
    n = len(rows)
    cutoff = int(TRAIN_FRAC * n)
    i_row = user_to_row[uid]

    for t, mid, r in rows[:cutoff]:
        j = movie_to_col[mid]
        k = rating_to_idx[float(r)]
        channel1[i_row, j, :] = 0.0
        channel1[i_row, j, k] = 1.0
        mask[i_row, j] = 1
        n_train_pairs += 1

    for t, mid, r in rows[cutoff:]:
        test_records.append({"userId": uid, "movieId": mid, "rating": r, "timestamp": t})
        n_test_pairs += 1

test_labels = pd.DataFrame(test_records, columns=["userId", "movieId", "rating", "timestamp"])

path_ch1 = out_dir / "channel1_softmax.npy"
path_mask = out_dir / "mask.npy"
path_test = out_dir / "test_labels.csv"

np.save(path_ch1, channel1)
np.save(path_mask, mask)
test_labels.to_csv(path_test, index=False)

print(f"Train pairs: {n_train_pairs:,}")
print(f"Test pairs:  {n_test_pairs:,}")
print(f"Saved: {path_ch1}  shape={channel1.shape} dtype={channel1.dtype}")
print(f"Saved: {path_mask}  shape={mask.shape} dtype={mask.dtype}")
print(f"Saved: {path_test}  rows={len(test_labels):,}")

del user_rows  # free memory

Streaming rating.csv for cohort × vocab ratings (+ timestamp) …
Train pairs: 4,119,189
Test pairs:  1,032,307
Saved: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/channel1_softmax.npy  shape=(5000, 13129, 10) dtype=float32
Saved: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/mask.npy  shape=(5000, 13129) dtype=int8
Saved: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/test_labels.csv  rows=1,032,307


## Part 4 — Training-set statistics for personality initialization

In [5]:
# Reconstruct train ratings from channel1 + mask
mu_user = np.zeros(n_users, dtype=np.float64)
train_sum_global = 0.0
train_cnt_global = 0

rating_from_idx = RATING_LEVELS  # index → rating value

for i in range(n_users):
    train_cols = np.where(mask[i] == 1)[0]
    assert len(train_cols) > 0, f"User row {i} (userId={cohort_user_ids[i]}) has zero train ratings"
    # one-hot → rating value
    ks = channel1[i, train_cols, :].argmax(axis=1)
    ratings_i = rating_from_idx[ks]
    mu_user[i] = float(ratings_i.mean())
    train_sum_global += float(ratings_i.sum())
    train_cnt_global += len(ratings_i)

mu_global = train_sum_global / train_cnt_global

path_means = out_dir / "user_means.npy"
np.save(path_means, mu_user)

n_generous = int(np.sum(mu_user > 4.0))
n_harsh = int(np.sum(mu_user < 2.5))

print(f"mu_global = {mu_global:.6f}")
print(
    f"mu_user: min={mu_user.min():.4f}, max={mu_user.max():.4f}, "
    f"median={float(np.median(mu_user)):.4f}, std={mu_user.std():.4f}"
)
print(f"Generous (mu_user > 4.0): {n_generous:,}")
print(f"Harsh    (mu_user < 2.5): {n_harsh:,}")
print(f"Saved: {path_means}  shape={mu_user.shape} dtype={mu_user.dtype}")

mu_global = 3.328083
mu_user: min=1.0831, max=4.9228, median=3.3852, std=0.4143
Generous (mu_user > 4.0): 236
Harsh    (mu_user < 2.5): 145
Saved: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/user_means.npy  shape=(5000,) dtype=float64


## Part 5 — Verification

In [6]:
print("=== Shapes ===")
print(f"  cohort_user_ids: {cohort_user_ids.shape}")
print(f"  movie_vocab:     {movie_vocab.shape}")
print(f"  channel1:        {channel1.shape}  {channel1.dtype}")
print(f"  mask:            {mask.shape}  {mask.dtype}")
print(f"  user_means:      {mu_user.shape}  {mu_user.dtype}")
print(f"  test_labels:     {len(test_labels):,} rows")

# One-hot constraint: every nonzero movie-column sums to exactly 1
col_sums = channel1.sum(axis=2)  # (n_users, n_movies)
nonzero = col_sums > 0
assert np.allclose(col_sums[nonzero], 1.0), "channel1 one-hot violated: nonzero columns must sum to 1"
print("✓ channel1 one-hot: every nonzero column sums to 1")

# Mask consistency
n_mask_ones = int(mask.sum())
n_ch1_nonzero = int(nonzero.sum())
assert n_mask_ones == n_ch1_nonzero, f"mask 1s ({n_mask_ones}) != channel1 nonzero cols ({n_ch1_nonzero})"
assert n_mask_ones == n_train_pairs, f"mask 1s ({n_mask_ones}) != n_train_pairs ({n_train_pairs})"
print(f"✓ mask consistency: {n_mask_ones:,} train cells")

# Test labels count
assert len(test_labels) == n_test_pairs, f"test_labels ({len(test_labels)}) != n_test_pairs ({n_test_pairs})"
print(f"✓ test_labels row count: {len(test_labels):,}")

# User 666 stats
if 666 in cohort_set:
    i666 = user_to_row[666]
    n_train_666 = int(mask[i666].sum())
    n_test_666 = int((test_labels["userId"] == 666).sum())
    print(f"\n=== User 666 ===")
    print(f"  train ratings: {n_train_666}")
    print(f"  test ratings:  {n_test_666}")
    print(f"  mu_user:       {mu_user[i666]:.4f}")
    # one example train movie
    j_ex = int(np.where(mask[i666] == 1)[0][0])
    mid_ex = int(movie_vocab[j_ex])
    r_ex = float(RATING_LEVELS[int(channel1[i666, j_ex].argmax())])
    print(f"  example train: movieId={mid_ex}, rating={r_ex}")
else:
    print("\nUser 666 NOT in cohort.")

sparsity = n_train_pairs / (n_users * n_movies)
print(f"\n=== Totals ===")
print(f"  train pairs: {n_train_pairs:,}")
print(f"  test pairs:  {n_test_pairs:,}")
print(f"  sparsity (train / (n_users × n_movies)): {sparsity:.6f}  ({100 * sparsity:.4f}%)")
print("\nAll checks passed.")

=== Shapes ===
  cohort_user_ids: (5000,)
  movie_vocab:     (13129,)
  channel1:        (5000, 13129, 10)  float32
  mask:            (5000, 13129)  int8
  user_means:      (5000,)  float64
  test_labels:     1,032,307 rows
✓ channel1 one-hot: every nonzero column sums to 1
✓ mask consistency: 4,119,189 train cells
✓ test_labels row count: 1,032,307

User 666 NOT in cohort.

=== Totals ===
  train pairs: 4,119,189
  test pairs:  1,032,307
  sparsity (train / (n_users × n_movies)): 0.062749  (6.2749%)

All checks passed.
